# Análise de Vendedores com Alta Quantidade de Vendas e Baixa Avaliação

Este notebook analisa vendedores da base Olist que possuem um volume relevante de vendas, mas com média de avaliação (review) baixa.  
O objetivo é identificar padrões e possíveis causas para esse comportamento.

**Fonte dos dados:** [Olist E-Commerce Public Dataset - Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)

---
## 1. Importação das bibliotecas

In [1]:
import pandas as pd
from pathlib import Path

---
## 2. Carregamento das tabelas

Carregamos apenas as tabelas necessárias para a análise:
- **order_items**: relaciona pedidos com vendedores e produtos
- **order_reviews**: contém as notas dadas pelos clientes
- **orders**: status e datas dos pedidos
- **sellers**: informações dos vendedores (cidade/estado)

In [4]:
import pandas as pd
from pathlib import Path

# Carregando as tabelas do dataset Olist
reviews = pd.read_csv("data/olist_order_reviews_dataset.csv")
order_items = pd.read_csv("data/olist_order_items_dataset.csv")
orders = pd.read_csv("data/olist_orders_dataset.csv")
sellers = pd.read_csv("data/olist_sellers_dataset.csv")
products = pd.read_csv("data/olist_products_dataset.csv")

print('Linhas carregadas:')
print(f'  orders:      {len(orders):>7,}')
print(f'  order_items: {len(order_items):>7,}')
print(f'  reviews:     {len(reviews):>7,}')
print(f'  products:    {len(products):>7,}')
print(f'  sellers:     {len(sellers):>7,}')

Linhas carregadas:
  orders:       99,441
  order_items: 112,650
  reviews:      99,224
  products:     32,951
  sellers:       3,095


---
## 3. Limpeza das tabelas

Cada tabela é limpa separadamente para facilitar a leitura e replicação.

In [5]:
# ── Limpeza: orders ──────────────────────────────────────────────────────────
# Mantemos apenas pedidos com status 'delivered' (entregues ao cliente)
# Pedidos cancelados, em processamento ou não entregues distorcem a análise de satisfação

orders_clean = orders[orders['order_status'] == 'delivered'].copy()

# Remove linhas sem data de entrega registrada (entrega não confirmada)
orders_clean = orders_clean.dropna(subset=['order_delivered_customer_date'])

print(f'orders original:  {len(orders):,} linhas')
print(f'orders limpo:     {len(orders_clean):,} linhas')

orders original:  99,441 linhas
orders limpo:     96,470 linhas


In [6]:
# ── Limpeza: reviews ─────────────────────────────────────────────────────────
# Remove linhas sem review_score (avaliação não registrada)
reviews_clean = reviews.dropna(subset=['review_score']).copy()

# Garante que review_score seja inteiro entre 1 e 5
reviews_clean = reviews_clean[reviews_clean['review_score'].between(1, 5)]

# Remove duplicatas: mesmo order_id com review duplicada (mantém a primeira)
reviews_clean = reviews_clean.drop_duplicates(subset='order_id', keep='first')

print(f'reviews original: {len(reviews):,} linhas')
print(f'reviews limpo:    {len(reviews_clean):,} linhas')

reviews original: 99,224 linhas
reviews limpo:    98,673 linhas


In [7]:
# ── Limpeza: order_items ─────────────────────────────────────────────────────
# Remove itens sem seller_id (não é possível identificar o vendedor)
items_clean = order_items.dropna(subset=['seller_id']).copy()

# Remove preços zerados ou negativos (registros inválidos)
items_clean = items_clean[items_clean['price'] > 0]

print(f'order_items original: {len(order_items):,} linhas')
print(f'order_items limpo:    {len(items_clean):,} linhas')

order_items original: 112,650 linhas
order_items limpo:    112,650 linhas


In [8]:
# ── Limpeza: sellers ─────────────────────────────────────────────────────────
# Remove vendedores sem cidade ou estado (dados incompletos de localização)
sellers_clean = sellers.dropna(subset=['seller_city', 'seller_state']).copy()

print(f'sellers original: {len(sellers):,} linhas')
print(f'sellers limpo:    {len(sellers_clean):,} linhas')

sellers original: 3,095 linhas
sellers limpo:    3,095 linhas


---
## 4. Montagem da base analítica de vendedores

Cruzamos as tabelas para calcular, por vendedor:
- Número total de pedidos entregues
- Média de avaliação (review_score)
- Receita total gerada

Em seguida, filtramos os vendedores que têm **muitas vendas e baixa nota**.

In [9]:
# Junta order_items com os pedidos entregues
items_delivered = items_clean.merge(
    orders_clean[['order_id']],
    on='order_id',
    how='inner'
)

# Junta com as avaliações
items_with_review = items_delivered.merge(
    reviews_clean[['order_id', 'review_score']],
    on='order_id',
    how='left'   # left para manter pedidos sem review (serão ignorados nas médias)
)

# Agrupamento por vendedor: contagem de pedidos únicos, média de nota e receita total
seller_stats = (
    items_with_review
    .groupby('seller_id')
    .agg(
        total_pedidos  = ('order_id',      'nunique'),
        media_nota     = ('review_score',  'mean'),
        receita_total  = ('price',         'sum')
    )
    .reset_index()
)


print(f'Total de vendedores com dados: {len(seller_stats):,}')
seller_stats.head()

Total de vendedores com dados: 2,970


,seller_id,total_pedidos,media_nota,receita_total
0,0015a82c2db000af6aaaf3ae2ecb0532,3,3.666667,2685.00
1,001cca7ae9ae17fb1caed9dfb1094831,195,3.965368,24487.03
2,002100f778ceb8431b7a1020ff7ab48f,50,4.037037,1216.60
3,003554e2dce176b5555353e4f3555ac8,1,5.000000,120.00
4,004c9cd9d87a3c30c522c48c4fc07416,156,4.132530,19569.73


In [10]:
# Foi criado uma nova coluna, onde todos os vendedores com nota <= 3 , são considerados detratores,

seller_stats ["detrator"] = seller_stats ["media_nota"].apply(lambda media: True if media <= 3 else False)

In [11]:
#criação de filtro pra dataframe detratores

detratores = seller_stats [seller_stats["detrator"] == True]


In [12]:
# Temos a visualização rapida de media, desvio, min, maximo e etc
detratores.describe()

,total_pedidos,media_nota,receita_total
count,271.000000,271.000000,271.000000
mean,5.184502,2.234119,1154.328967
std,16.077218,0.800038,3695.091642
min,1.000000,1.000000,6.900000
25%,1.000000,1.400000,85.425000
50%,2.000000,2.500000,199.000000
75%,4.000000,3.000000,753.380000
max,187.000000,3.000000,38990.720000


In [13]:
#fazemos a mesma analise para o dataframe seller_stats

seller_stats [seller_stats["detrator"] == False].describe()

,total_pedidos,media_nota,receita_total
count,2699.000000,2694.000000,2699.000000
mean,35.719155,4.342534,4782.299289
std,110.070338,0.483247,14521.367738
min,1.000000,3.034483,6.500000
25%,3.000000,4.000000,266.000000
50%,8.000000,4.333333,987.700000
75%,26.000000,4.750000,3792.500000
max,1819.000000,5.000000,226987.930000


In [14]:
#Visualizamos quantos vendedores são detratores e quantos não são

seller_stats["detrator"].value_counts()

detrator
False    2699
True      271
Name: count, dtype: int64

In [15]:
#Criação do nosso novo dataframe 
final_detratores = (
    seller_stats
    .groupby('detrator')
    .agg(
        total_pedidos  = ('total_pedidos',      'sum'),
        receita_total  = ('receita_total',         'sum'),
        vendedores = ("seller_id", "count")
    )
    .reset_index()
)

In [16]:
# criação do total de receita e a porcentagem desse total de receita
total_receita =  final_detratores["receita_total"].sum()
final_detratores["%Receita_total"] =(final_detratores["receita_total"]/total_receita)*100

In [17]:
# criação do total de pedido e a porcentagem desse total de pedido

total_pedido =  final_detratores["total_pedidos"].sum()
final_detratores["%Pedido_total"] =(final_detratores["total_pedidos"]/total_pedido)*100

In [18]:
# criação do total de vendedores e a porcentagem desse total de vendedores

total_vendedores =  final_detratores["vendedores"].sum()
final_detratores["%Vendedor_total"] =(final_detratores["vendedores"]/total_vendedores)*100

In [19]:
final_detratores

,detrator,total_pedidos,receita_total,vendedores,%Receita_total,%Pedido_total,%Vendedor_total
0,False,96406,12907425.78,2699,97.633757,98.563556,90.875421
1,True,1405,312823.15,271,2.366243,1.436444,9.124579


- Narrativa de que 10% dos vendedores são considerados detratores, porém impactam em 1% da receita.
- Graficos mostram a receita, vendedores, impacto em prcentagem